# 09｜YOLO 整体结构：Backbone、Neck 与 Detection Head

上一课用简化 YOLO 走完了训练、推理和评价流程。这一课开始正式认识 YOLO 网络本身，只解决一个核心问题：

**图片进入 YOLO 后，Backbone、Neck 和 Detection Head 分别做什么？**

暂时不学习 Anchor、坐标解码公式、具体目标分配算法和版本差异。先把整座房子的结构看清楚，再进入房间研究细节。

## 一、YOLO 为什么叫一阶段检测器

两阶段检测器可以粗略理解为先找候选区域，再对候选区域分类并修正边界框：

$$
\text{先提出候选区域}
\longrightarrow \text{再分类并修正框}
$$

YOLO 类方法则在同一个端到端检测网络中，直接从特征图产生类别和边界框：

$$
\text{图片}
\longrightarrow \text{一个检测网络}
\longrightarrow \text{大量候选检测结果}
$$

“一阶段”不是说网络只有一层，也不是只做一次卷积，而是没有一个独立的候选区域阶段再接第二套检测阶段。

## 二、先记住整条网络主线

一个常见的 YOLO 风格网络可以抽象为：

$$
\text{输入图片}
\xrightarrow{\text{Backbone}}
\text{多层图像特征}
\xrightarrow{\text{Neck}}
\text{融合后的多尺度特征}
\xrightarrow{\text{Detection Head}}
\text{类别、目标性与边界框}
$$

翻译成最简单的话：

- Backbone：从图片中提取有用信息；
- Neck：让不同大小、不同深浅的特征互相交流；
- Detection Head：把整理好的特征变成检测答案。

它们不是三个互不相关的模型，而是同一个网络前后相接的三个功能区。

## 三、Backbone：把像素变成越来越抽象的特征

Backbone 就是骨干网络。它通过卷积、激活、下采样等操作，把原始像素逐层加工成特征图。

浅层特征容易保留边缘、纹理和精细位置；深层特征更容易表达眼睛、耳朵、物体轮廓和整体语义。

Backbone 并不直接宣布“这里有狗”。它先准备判断所需的信息，后面的检测头才根据这些信息作答。

假设输入图片大小为 $640\times640$，空间尺寸可能逐层变化：

$$
640\times640
\longrightarrow 320\times320
\longrightarrow 160\times160
\longrightarrow 80\times80
\longrightarrow 40\times40
\longrightarrow 20\times20
$$

越往后空间尺寸通常越小，计算量下降，一个位置整合的原图范围也会扩大；代价是小目标的精细位置可能逐渐减弱。

## 四、用三个 Backbone 输出建立直觉

为了教学，假设 Backbone 从不同深度取出三张特征图：

| 特征图 | 示例形状 | 相对特点 |
|---|---|---|
| $F_3$ | $B\times256\times80\times80$ | 位置较细，保留较多小目标信息 |
| $F_4$ | $B\times512\times40\times40$ | 细节与语义居中 |
| $F_5$ | $B\times1024\times20\times20$ | 感受野较大，语义较强 |

$B$ 是 batch 大小，第二个维度是通道数，最后两个维度是高和宽。这些数字只是示例，不是所有 YOLO 的固定配置。

如果 $640\times640$ 图片变成 $80\times80$ 特征图，空间尺寸缩小 8 倍，常说这张特征图的 Stride 为 8：

$$
\operatorname{stride}=\frac{640}{80}=8
$$

Stride 帮助把特征图位置映射回原图的大致位置，但不代表一个位置只能看固定的 $8\times8$ 小块。它的感受野通常更大。

## 五、为什么只使用最深层特征还不够

最深层特征语义很强，可能很会判断“这里像一只狗”，但它的空间尺寸较小。对于只占十几个像素的小猫，连续下采样后，位置信息可能已经很弱。

较浅层特征位置更密，容易保留小猫在哪里，却可能缺少足够强的类别语义。

检测因此面临一个矛盾：

$$
\begin{aligned}
\text{浅层特征}&:\ \text{位置细，但语义相对弱}\\
\text{深层特征}&:\ \text{语义强，但位置相对粗}
\end{aligned}
$$

Neck 就是为了让这两类信息相互帮助。

## 六、Neck：融合不同尺度的信息

Neck 位于 Backbone 和 Detection Head 之间。它通过上采样、下采样、拼接或相加等操作融合不同深度的特征。

可以把 Neck 想成信息交流站：

- 把深层的物体语义传给高分辨率特征；
- 把浅层的空间细节带到其他尺度；
- 生成更适合检测头使用的多尺度特征。

$$
(F_3,F_4,F_5)
\xrightarrow{\text{多尺度特征融合}}
(P_3,P_4,P_5)
$$

$F$ 表示 Backbone 提取的特征，$P$ 表示 Neck 整理后交给检测头的特征。

上采样本身不会凭空创造真实细节。它的作用是先把特征调整到可以对齐的尺寸，再让深层语义与浅层细节结合。

## 七、Detection Head：把特征变成检测答案

Detection Head，也就是检测头，会读取 $P_3$、$P_4$、$P_5$ 上各个位置的特征向量，产生候选预测。

$$
z_{ij}
\xrightarrow{\text{Detection Head}}
\text{目标性、类别分数、边界框参数}
$$

Backbone 和 Neck 产生的仍是内部特征。直到 Head 输出这些数字，模型才真正提出“这里可能有一只狗，框大约在这里”。

预测槽位是输出表中的固定位置，Detection Head 是产生该位置输出的网络模块：

$$
\text{某个特征图位置的向量}
\xrightarrow{\text{检测头}}
\text{该位置对应槽位的检测输出}
$$

槽位说明输出在哪里；检测头说明输出怎样由特征计算出来。

## 八、三个尺度一共会产生多少候选

假设使用 $80\times80$、$40\times40$、$20\times20$ 三个输出尺度，并且每个位置只有一个预测单位：

$$
\begin{aligned}
N
&=80\times80+40\times40+20\times20\\
&=6400+1600+400\\
&=8400
\end{aligned}
$$

一张图片可能只有 3 个真实物体，模型却准备了 8400 个候选位置。绝大多数位置应该判断为背景，少数位置会给出高分物体预测。

所以 8400 不是模型认为图中有 8400 个物体，而是模型准备了 8400 个提出候选的机会。

## 九、不同尺度为什么适合不同大小的目标

高分辨率特征图位置更多、更密，原图中的小目标更容易留下可分辨的空间信息。低分辨率特征图位置较少，但每个位置整合的范围更大，语义通常更强。

可以先建立这个直觉：

$$
\begin{aligned}
P_3\ (80\times80)&:\ \text{更有利于小目标与精细位置}\\
P_4\ (40\times40)&:\ \text{处理中等尺度目标}\\
P_5\ (20\times20)&:\ \text{更有利于大目标与强语义}
\end{aligned}
$$

这只是主要倾向，不是硬性规定。最终责任还会受网络结构和目标分配规则影响。

## 十、跟踪一只小猫走完整条路线

假设图片远处有一只很小的猫：

1. Backbone 的浅层保留猫所在位置的细节，深层逐渐形成“像动物”的语义；
2. Neck 把深层语义与高分辨率特征融合；
3. Detection Head 在高分辨率尺度的一些位置读取融合后的特征；
4. 这些位置输出目标性、猫类别分数和边界框参数；
5. 训练时，目标分配选出负责单位并计算损失；
6. 推理时，分数筛选和 NMS 从候选中留下最终猫框。

这就是三部分的连续关系：Backbone 形成信息，Neck 组合信息，Head 使用信息作答。

## 十一、损失怎样训练三个部分

检测损失在网络末端计算，但梯度不会只停留在 Detection Head：

$$
\text{检测损失}
\longrightarrow \text{Detection Head}
\longrightarrow \text{Neck}
\longrightarrow \text{Backbone}
$$

因此 Head 学习怎样输出类别与框，Neck 学习怎样融合信息，Backbone 学习怎样从像素提取更适合检测的特征。三部分通常通过同一个总损失端到端协同学习。

## 十二、网络结构与其他规则不要混在一起

| 类型 | 包含什么 | 核心作用 |
|---|---|---|
| 网络结构 | Backbone、Neck、Detection Head | 从图片计算检测预测 |
| 训练规则 | 目标分配、损失、反向传播 | 建立监督并更新参数 |
| 推理后处理 | 分数筛选、NMS | 整理网络候选 |
| 评价规则 | IoU 匹配、AP、mAP | 衡量预测质量 |

目标分配不是 Neck，也不是 Head 后面的另一层网络；NMS 也不属于 Backbone。它们围绕网络输出工作，但职责完全不同。

## 十三、完整前向流程再看一次

$$
\begin{aligned}
\text{图片}
&\xrightarrow{\text{Backbone}}
(F_3,F_4,F_5)\\
&\xrightarrow{\text{Neck}}
(P_3,P_4,P_5)\\
&\xrightarrow{\text{Detection Head}}
\text{三个尺度上的候选预测}\\
&\xrightarrow{\text{解码、分数筛选、NMS}}
\text{最终检测结果}
\end{aligned}
$$

前三步属于神经网络的前向计算；解码、筛选和 NMS 属于把原始输出整理成可用结果的过程。

## 十四、常见误区

### 误区 1：Backbone 已经输出类别和边界框

Backbone 主要输出特征图。真正把特征转成检测数字的是 Detection Head。

### 误区 2：Neck 只是把特征图放大

上采样只是手段之一。Neck 的核心目的是对齐并融合不同深度、不同尺度的信息。

### 误区 3：Detection Head 是一个神经元

它通常是一组可学习层，会作用在大量位置上，同时产生许多候选预测。

### 误区 4：$80\times80$ 特征图代表检测到 6400 个物体

它只提供 6400 个候选位置，大部分位置应判断为背景。

### 误区 5：一阶段表示网络只计算一步

YOLO 内部仍有很多层；一阶段指它直接产生检测候选，不先运行一个独立的候选区域检测阶段。

## 十五、自我检查

1. Backbone、Neck、Detection Head 各自负责什么？
2. 为什么 Backbone 越深，特征图空间尺寸通常越小？
3. 为什么检测不能只依赖最深层特征？
4. Neck 中的上采样为什么不等于凭空创造细节？
5. 三个输出尺度各有什么主要倾向？
6. 8400 个候选位置是否表示图中有 8400 个物体？
7. 预测槽位和 Detection Head 有什么区别？
8. 检测损失为什么也能训练 Backbone？

### 参考答案

1. Backbone 提取多层特征，Neck 融合多尺度信息，Head 把特征变成目标性、类别和边界框。
2. 下采样能降低计算量并扩大后层位置整合的原图范围，但会减少空间细节。
3. 最深层语义强但位置较粗，小目标信息可能不足。
4. 上采样主要用来对齐尺寸；真正有价值的是深层语义与浅层细节的融合。
5. 高分辨率尺度更有利于小目标，中间尺度居中，低分辨率尺度更有利于大目标。
6. 不是。它表示模型有 8400 个提出候选的机会。
7. 槽位是固定输出位置，Head 是从特征计算该位置输出的网络模块。
8. 反向传播会让梯度从损失经过 Head、Neck，继续传到 Backbone。

## 十六、本课总结

$$
\underbrace{\text{Backbone}}_{\text{提取不同深度的图像特征}}
\longrightarrow
\underbrace{\text{Neck}}_{\text{融合不同尺度的信息}}
\longrightarrow
\underbrace{\text{Detection Head}}_{\text{输出目标性、类别和边界框}}
$$

Backbone 解决“图片中有什么信息”，Neck 解决“不同层次的信息怎样结合”，Head 解决“怎样把信息变成检测答案”。

## 下一课：多尺度输出怎样变成边界框

下一课继续使用一个小网格和一只狗，回答：

- 特征图位置怎样对应回原图坐标？
- Stride 在坐标映射中起什么作用？
- 检测头输出的数字怎样变成完整边界框？
- Anchor-based 和 Anchor-free 最基本的区别是什么？